# Phase 3 — Leakage-safe temporal features

Each row represents a failed payment and contains only information from the same customer's transactions with `history_timestamp < prediction_time`. The current payment, same-timestamp payments, full-history customer profiles, synthetic archetypes, recovery labels, and future events are excluded.

In [ ]:
from pathlib import Path
import json
import pandas as pd


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        path = candidate / "ml" / "data" / "processed" / "failed_payment_features.csv"
        if path.exists():
            return candidate
    raise FileNotFoundError("Run `python -m ml.src.build_features` first")


ROOT = find_repo_root()
PROCESSED = ROOT / "ml" / "data" / "processed"
features = pd.read_csv(PROCESSED / "failed_payment_features.csv", parse_dates=["prediction_time"])
transactions = pd.read_csv(PROCESSED / "transactions_with_customers.csv", parse_dates=["timestamp"])
summary = json.loads((PROCESSED / "temporal_feature_summary.json").read_text())

print("Feature shape:", features.shape)
summary

In [ ]:
print(features.columns.tolist())
display(features.head(3))

print("Missing values:", int(features.isna().sum().sum()))
print("Duplicate payment rows:", int(features["transaction_id"].duplicated().sum()))

In [ ]:
columns = [
    "previous_transaction_count",
    "historical_success_rate",
    "previous_failure_streak",
    "failures_last_7d",
    "failures_last_30d",
    "amount_vs_previous_avg",
]

print(features[columns].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))
features[columns].hist(figsize=(14, 8), bins=35)

In [ ]:
failed_ids = set(
    transactions.loc[transactions["transaction_status"].eq("FAILED"), "transaction_id"]
)
assert len(features) == 12_376
assert features["transaction_id"].is_unique
assert set(features["transaction_id"]) == failed_ids
assert features["previous_transaction_count"].eq(
    features["previous_success_count"] + features["previous_failure_count"]
).all()
assert summary["temporal_leakage_violations"] == 0
assert summary["rolling_window_violations"] == 0
print("Failed-payment population and historical counts are complete.")

In [ ]:
print("Rows without prior history:", int(features["has_prior_history"].eq(0).sum()))
print("Rows without prior success:", int(features["has_previous_success"].eq(0).sum()))
print("Rows without prior failure:", int(features["has_previous_failure"].eq(0).sum()))

sentinel_rows = features.loc[features["has_prior_history"].eq(0)]
assert sentinel_rows["previous_transaction_count"].eq(0).all()
assert sentinel_rows["days_since_previous_transaction"].eq(-1).all()
assert sentinel_rows["customer_primary_device_before_failure"].eq("UNKNOWN").all()
sentinel_rows.head(3)

In [ ]:
sample = features.loc[features["previous_transaction_count"].gt(0)].iloc[0]
history = transactions.loc[
    transactions["customer_id"].eq(sample["customer_id"])
    & transactions["timestamp"].lt(sample["prediction_time"])
].sort_values(["timestamp", "transaction_id"])

assert len(history) == sample["previous_transaction_count"]
assert history["timestamp"].max() < sample["prediction_time"]
assert sample["transaction_id"] not in set(history["transaction_id"])

print("Prediction payment:")
display(sample.to_frame("value"))
print("Strictly prior history:")
display(history.tail(10))

In [ ]:
print("Device match rate:", features["device_matches_primary"].mean())
print("Network match rate:", features["network_matches_primary"].mean())
print("Fraud-flagged failures:", int(features["fraud_flag"].sum()))

context_summary = features.groupby("has_prior_history").agg(
    rows=("transaction_id", "size"),
    median_history=("previous_transaction_count", "median"),
    median_success_rate=("historical_success_rate", "median"),
    median_amount_ratio=("amount_vs_previous_avg", "median"),
)
context_summary

## Feature semantics

- Zero-valued historical statistics mean there was no prior history; use the corresponding `has_*` indicator.
- Recency uses `-1` days when no qualifying prior transaction exists.
- Historical categorical modes use `UNKNOWN` without prior history.
- Seven- and thirty-day windows use strict lower and upper bounds.
- These rows contain predictors only. There is no `recovered` target, intervention outcome, or model score yet.
- Phase 4 must preserve these feature definitions when designing the synthetic recovery environment.